# LRDB â€” MongoDB EDA (Logistics Relational DB)

**Goal:** understand the core of the data before building bronze/silver/gold. Covers connection via `.env`, collection inventory, schema, data quality, distributions, time coverage, FK integrity, and lakehouse mapping.

- System of record: MongoDB 7 replicaSet `rs0` (14 collections, ~570k docs)
- Run: `pip install pymongo python-dotenv pandas matplotlib seaborn plotly` then `jupyter notebook`
- `.env` is the single source for `MONGO_URI` / `MONGO_DB` (see `../.env` and `../.env.example`)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pymongo import MongoClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

ROOT = Path.cwd()
# handle running from notebooks/ or repo root
for p in [ROOT, ROOT.parent, Path("C:/Git-Repo/LRDB")]:
    if (p / ".env").exists():
        load_dotenv(p / ".env", override=True)
        ROOT = p
        break
else:
    load_dotenv(".env")
print("ROOT", ROOT)
print("MONGO_URI", os.getenv("MONGO_URI"))
print("MONGO_DB", os.getenv("MONGO_DB"))
print("Exists .env:", (ROOT / ".env").exists())

In [ ]:
def get_client(uri=None, db_name=None):
    uri = uri or os.getenv("MONGO_URI", "mongodb://localhost:27017")
    db_name = db_name or os.getenv("MONGO_DB", "LRDB")
    # try with replicaSet first, fallback without
    for cand in [
        uri,
        "mongodb://localhost:27017",
        "mongodb://localhost:27017/?replicaSet=rs0",
    ]:
        try:
            c = MongoClient(cand, serverSelectionTimeoutMS=3000)
            c.admin.command("ping")
            print(f"Connected via: {cand}")
            return c, c[db_name]
        except Exception as e:
            print(f"Failed {cand}: {e}")
    raise RuntimeError(
        "Cannot connect to MongoDB - is docker compose up? run: make up PROFILES=core"
    )


client, db = get_client()
print("DB:", db.name)
cols = sorted(db.list_collection_names())
print(f"Collections ({len(cols)}):", cols)

In [ ]:
# 1. Inventory - counts, sizes, avg doc
rows = []
for n in cols:
    coll = db[n]
    cnt = coll.count_documents({})
    stats = db.command("collStats", n)
    rows.append(
        {
            "collection": n,
            "count": cnt,
            "size_MB": round(stats.get("size", 0) / 1024 / 1024, 2),
            "avgObjSize_B": round(stats.get("avgObjSize", 0), 1),
            "storage_MB": round(stats.get("storageSize", 0) / 1024 / 1024, 2),
        }
    )
inv = pd.DataFrame(rows).sort_values("count", ascending=False)
print(inv.to_string(index=False))
print(
    f"\nTotal docs: {inv['count'].sum():,} | Total size: {inv['size_MB'].sum():.1f} MB"
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=inv, x="collection", y="count", palette="viridis", ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_title("Document count per collection")
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height()):,}",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=8,
    )
plt.tight_layout()
plt.show()

In [ ]:
# 2. Schema per collection - keys + sampled types
def schema_for(coll_name, sample_n=3):
    coll = db[coll_name]
    doc = coll.find_one()
    keys = sorted(doc.keys()) if doc else []
    samples = list(coll.find().limit(sample_n))
    return keys, doc, samples


for n in cols:
    keys, doc, samples = schema_for(n)
    print(f"\n=== {n} ({db[n].count_documents({}):,} docs) ===")
    print("Keys:", ", ".join(keys))
    for k in keys:
        v = doc.get(k)
        t = type(v).__name__
        preview = str(v)[:90].replace(chr(10), " ")
        print(f"  {k:24} {t:12} e.g. {preview}")

In [ ]:
# 3. Data quality - null/empty, duplicates, updated_at coverage
def quality_report(coll_name, sample_size=5000):
    coll = db[coll_name]
    total = coll.count_documents({})
    n = min(sample_size, total)
    docs = list(coll.find().limit(n))
    if not docs:
        return pd.DataFrame()
    keys = sorted({k for d in docs for k in d.keys()} - {"_id"})
    recs = []
    for k in keys:
        empty = sum(1 for d in docs if d.get(k) in [None, "", "null", "NULL"])
        recs.append(
            {"field": k, "empty_or_null": empty, "pct_empty": round(empty / n * 100, 2)}
        )
    # duplicate check on business key (first non-_id)
    bkeys = {
        "loads": "load_id",
        "trips": "trip_id",
        "customers": "customer_id",
        "drivers": "driver_id",
        "trucks": "truck_id",
        "routes": "route_id",
        "facilities": "facility_id",
        "trailers": "trailer_id",
        "delivery_events": "event_id",
        "fuel_purchases": "fuel_purchase_id",
        "safety_incidents": "incident_id",
        "maintenance_records": "maintenance_id",
    }
    bkey = bkeys.get(coll_name)
    dup_info = ""
    if bkey:
        distinct = len(coll.distinct(bkey))
        dup_info = f" | {bkey} distinct={distinct:,} dup={total-distinct:,}"
    print(f"{coll_name}: n={total:,}{dup_info}")
    return pd.DataFrame(recs).sort_values("pct_empty", ascending=False)


for n in cols:
    dfq = quality_report(n)
    if not dfq.empty and (dfq["pct_empty"] > 0).any():
        display(dfq[dfq["pct_empty"] > 0])
    else:
        print("  -> no empty/null fields in sample")
    print()

In [ ]:
# 4. Temporal coverage - min/max dates per collection
def temporal_coverage():
    recs = []
    mapping = [
        ("loads", "load_date"),
        ("trips", "dispatch_date"),
        ("fuel_purchases", "purchase_date"),
        ("delivery_events", "scheduled_datetime"),
        ("maintenance_records", "maintenance_date"),
        ("safety_incidents", "incident_date"),
        ("drivers", "hire_date"),
        ("customers", "contract_start_date"),
    ]
    for coll, field in mapping:
        try:
            mn = list(db[coll].find().sort(field, 1).limit(1))
            mx = list(db[coll].find().sort(field, -1).limit(1))
            recs.append(
                {
                    "collection": coll,
                    "field": field,
                    "min": mn[0][field] if mn else None,
                    "max": mx[0][field] if mx else None,
                    "count": db[coll].count_documents({}),
                }
            )
        except Exception as e:
            recs.append({"collection": coll, "field": field, "min": str(e)})
    return pd.DataFrame(recs)


tdf = temporal_coverage()
display(tdf)
print(
    "\nAll collections share updated_at ~ 2026-06-16 (bulk load). Business dates span 2022-01-01 to 2024-12/2025-01."
)


# Monthly volume for loads + trips
def monthly_volume(coll, date_field):
    pipeline = [
        {"$group": {"_id": {"$substr": [f"${date_field}", 0, 7]}, "cnt": {"$sum": 1}}},
        {"$sort": {"_id": 1}},
    ]
    try:
        res = list(db[coll].aggregate(pipeline))
        return pd.DataFrame(
            [(r["_id"], r["cnt"]) for r in res], columns=["month", "count"]
        )
    except Exception as e:
        print(e)
        return pd.DataFrame()


fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (coll, fld) in zip(axes, [("loads", "load_date"), ("trips", "dispatch_date")]):
    mdf = monthly_volume(coll, fld)
    if not mdf.empty:
        ax.plot(mdf["month"], mdf["count"])
        ax.set_title(f"{coll} monthly volume ({fld})")
        ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()
if not monthly_volume("loads", "load_date").empty:
    display(monthly_volume("loads", "load_date").head(10))
    display(monthly_volume("loads", "load_date").tail(10))

In [ ]:
# 5. Core business distributions - loads (the revenue fact)
SAMPLE = int(os.getenv("EDA_SAMPLE_SIZE", "5000"))
print("EDA_SAMPLE_SIZE", SAMPLE)
loads_df = pd.DataFrame(list(db.loads.find().limit(20000)))
loads_df["load_date"] = pd.to_datetime(loads_df["load_date"], errors="coerce")
print(loads_df.shape)
display(loads_df.describe(include="all").T)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
# revenue
axes[0].hist(loads_df["revenue"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("loads.revenue")
axes[1].hist(loads_df["weight_lbs"], bins=50, color="darkorange", edgecolor="white")
axes[1].set_title("weight_lbs")
axes[2].hist(loads_df["fuel_surcharge"], bins=50, color="seagreen", edgecolor="white")
axes[2].set_title("fuel_surcharge")
axes[3].hist(loads_df["pieces"], bins=30, color="purple", edgecolor="white")
axes[3].set_title("pieces")
axes[4].hist(
    loads_df["accessorial_charges"], bins=30, color="crimson", edgecolor="white"
)
axes[4].set_title("accessorial_charges (many zeros)")
# revenue vs weight
axes[5].scatter(loads_df["weight_lbs"], loads_df["revenue"], alpha=0.1, s=5)
axes[5].set_title("revenue vs weight")
axes[5].set_xlabel("weight_lbs")
axes[5].set_ylabel("revenue")
plt.tight_layout()
plt.show()

# categoricals
for col in ["load_status", "booking_type", "load_type"]:
    print(f"\n{col}:")
    print(loads_df[col].value_counts().to_string())
    plt.figure(figsize=(6, 3))
    loads_df[col].value_counts().plot(kind="bar", color="teal", edgecolor="white")
    plt.title(col)
    plt.tight_layout()
    plt.show()

In [ ]:
# 6. Trips + delivery_events
trips_df = pd.DataFrame(list(db.trips.find().limit(20000)))
print("trips", trips_df.shape)
display(trips_df.describe().T)
print(triips_df_missing := trips_df.isin(["", "null"]).sum())
print(
    "\ntrips empty driver_id:", (trips_df["driver_id"] == "").sum(), "/", len(trips_df)
)
print("empty truck_id:", (trips_df["truck_id"] == "").sum())
print("empty trailer_id:", (trips_df["trailer_id"] == "").sum())
print(trips_df["trip_status"].value_counts())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(
    trips_df["actual_distance_miles"], bins=50, color="navy", edgecolor="white"
)
axes[0].set_title("actual_distance_miles")
axes[1].hist(trips_df["fuel_gallons_used"], bins=50, color="olive", edgecolor="white")
axes[1].set_title("fuel_gallons_used")
axes[2].hist(trips_df["average_mpg"], bins=50, color="tomato", edgecolor="white")
axes[2].set_title("average_mpg")
plt.tight_layout()
plt.show()

ev_df = pd.DataFrame(list(db.delivery_events.find().limit(20000)))
print(
    "delivery_events",
    ev_df.shape,
    ev_df["event_type"].value_counts().to_dict(),
    ev_df["on_time_flag"].value_counts().to_dict(),
)
ev_df["detention_minutes"] = pd.to_numeric(ev_df["detention_minutes"], errors="coerce")
plt.figure(figsize=(8, 4))
plt.hist(ev_df["detention_minutes"], bins=60, color="slateblue", edgecolor="white")
plt.title("detention_minutes (delivery_events)")
plt.show()
print(ev_df["detention_minutes"].describe().to_string())
display(ev_df.groupby(["event_type", "on_time_flag"]).size().unstack(fill_value=0))

In [ ]:
# 7. Fuel purchases - anomalies (location_state mismatch exposed)
fuel_df = pd.DataFrame(list(db.fuel_purchases.find().limit(20000)))
fuel_df["purchase_date"] = pd.to_datetime(fuel_df["purchase_date"], errors="coerce")
fuel_df["gallons"] = pd.to_numeric(fuel_df["gallons"], errors="coerce")
fuel_df["price_per_gallon"] = pd.to_numeric(
    fuel_df["price_per_gallon"], errors="coerce"
)
print(fuel_df.shape, "empty driver_id", (fuel_df["driver_id"] == "").sum())
display(fuel_df[["gallons", "price_per_gallon", "total_cost"]].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(fuel_df["gallons"], bins=50, color="chocolate", edgecolor="white")
axes[0].set_title("gallons")
axes[1].hist(
    fuel_df["price_per_gallon"], bins=50, color="forestgreen", edgecolor="white"
)
axes[1].set_title("price_per_gallon")
plt.tight_layout()
plt.show()

# city vs state mismatch eg New York, AZ
print("Top city/state combos:")
display(
    fuel_df.groupby(["location_city", "location_state"])
    .size()
    .sort_values(ascending=False)
    .head(15)
)
print("\nSuspicious: New York, AZ appears - likely synthetic data bug")
print(
    fuel_df[fuel_df["location_city"] == "New York"]["location_state"]
    .value_counts()
    .head()
)

In [ ]:
# 8. Dimensions - customers, routes, trucks, drivers, facilities
cust_df = pd.DataFrame(list(db.customers.find()))
print("customers", cust_df.shape)
display(cust_df["customer_type"].value_counts())
display(cust_df["account_status"].value_counts())
plt.figure(figsize=(7, 3))
cust_df["annual_revenue_potential"].hist(bins=30, color="goldenrod", edgecolor="white")
plt.title("customers.annual_revenue_potential")
plt.show()

trucks_df = pd.DataFrame(list(db.trucks.find()))
print(
    "trucks",
    trucks_df.shape,
    trucks_df["status"].value_counts().to_dict(),
    trucks_df["make"].value_counts().to_dict(),
)

drivers_df = pd.DataFrame(list(db.drivers.find()))
print(
    "drivers",
    drivers_df.shape,
    drivers_df["employment_status"].value_counts().to_dict(),
)
print("termination_date empty", (drivers_df["termination_date"] == "").sum(), "/150")
plt.figure(figsize=(6, 3))
plt.hist(
    drivers_df["years_experience"], bins=15, color="mediumpurple", edgecolor="white"
)
plt.title("drivers.years_experience")
plt.show()

routes_df = pd.DataFrame(list(db.routes.find()))
print("routes", routes_df.shape)
display(
    routes_df[
        [
            "typical_distance_miles",
            "base_rate_per_mile",
            "fuel_surcharge_rate",
            "typical_transit_days",
        ]
    ]
    .describe()
    .T
)

fac_df = pd.DataFrame(list(db.facilities.find()))
print("facilities", fac_df.shape, fac_df["facility_type"].value_counts().to_dict())
plt.figure(figsize=(6, 3))
plt.scatter(fac_df["longitude"], fac_df["latitude"], c="red", alpha=0.6)
plt.title("facilities geo")
plt.xlabel("lon")
plt.ylabel("lat")
plt.show()
display(fac_df.head(3))

In [ ]:
# 9. FK integrity - the relational core (LRDB)
checks = []
cust_ids = set(db.customers.distinct("customer_id"))
route_ids = set(db.routes.distinct("route_id"))
load_ids = set(db.loads.distinct("load_id"))
trip_ids = set(db.trips.distinct("trip_id"))
truck_ids = set(db.trucks.distinct("truck_id"))
driver_ids = set(db.drivers.distinct("driver_id"))


def fk_check(coll, field, valid_set):
    distinct = set(db[coll].distinct(field))
    # ignore empty strings for optional FKs
    distinct_nz = {x for x in distinct if x not in ("", None)}
    missing = distinct_nz - valid_set
    return {
        "collection": coll,
        "field": field,
        "distinct": len(distinct_nz),
        "missing": len(missing),
        "missing_sample": list(missing)[:3],
    }


checks.append(fk_check("loads", "customer_id", cust_ids))
checks.append(fk_check("loads", "route_id", route_ids))
checks.append(fk_check("trips", "load_id", load_ids))
checks.append(fk_check("trips", "truck_id", truck_ids))
checks.append(fk_check("trips", "driver_id", driver_ids))
checks.append(fk_check("delivery_events", "trip_id", trip_ids))
checks.append(fk_check("delivery_events", "load_id", load_ids))
checks.append(fk_check("fuel_purchases", "trip_id", trip_ids))
checks.append(fk_check("fuel_purchases", "truck_id", truck_ids))
checks.append(fk_check("safety_incidents", "trip_id", trip_ids))
checks.append(fk_check("maintenance_records", "truck_id", truck_ids))

fk_df = pd.DataFrame(checks)
display(fk_df)
print(
    "\nAll core FKs resolve (0 missing) -> relational model is consistent. Optional FKs with empty strings are expected."
)
print(
    f"loads 1:1 trips? loads={len(load_ids):,} trips distinct load_id={len(set(db.trips.distinct('load_id'))):,}"
)
print(
    f"delivery_events 2 per load? {db.delivery_events.count_documents({})/db.loads.count_documents({}):.1f}"
)
print(
    f"fuel per trip? {db.fuel_purchases.count_documents({})/db.trips.count_documents({}):.1f}"
)

In [ ]:
# 10. Correlations & business signals
loads_df["revenue_per_mile"] = (
    loads_df["revenue"] / loads_df["weight_lbs"].replace(0, np.nan) * 1000
)
corr_cols = ["weight_lbs", "pieces", "revenue", "fuel_surcharge", "accessorial_charges"]
corr = loads_df[corr_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("loads numeric correlations")
plt.tight_layout()
plt.show()

# trips correlation
tcorr = trips_df[
    [
        "actual_distance_miles",
        "actual_duration_hours",
        "fuel_gallons_used",
        "average_mpg",
        "idle_time_hours",
    ]
].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(tcorr, annot=True, cmap="viridis", fmt=".2f")
plt.title("trips correlations")
plt.tight_layout()
plt.show()

# On-time rate over time (delivery_events)
ev_all = pd.DataFrame(
    list(db.delivery_events.find({}, {"scheduled_datetime": 1, "on_time_flag": 1}))
)
ev_all["month"] = ev_all["scheduled_datetime"].str.slice(0, 7)
otr = (
    ev_all.groupby("month")["on_time_flag"]
    .apply(lambda s: (s == "True").mean())
    .reset_index()
)
otr.columns = ["month", "on_time_rate"]
plt.figure(figsize=(12, 4))
plt.plot(otr["month"], otr["on_time_rate"])
plt.xticks(rotation=45)
plt.title("On-time delivery rate by month (delivery_events)")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()
display(otr.head(10))

# Revenue per customer (top 15)
rev_by_cust = (
    loads_df.groupby("customer_id")["revenue"]
    .agg(["sum", "count", "mean"])
    .sort_values("sum", ascending=False)
    .head(15)
)
print("Top customers by revenue:")
display(rev_by_cust)
rev_by_cust["sum"].plot(kind="bar", figsize=(12, 4), color="teal", edgecolor="white")
plt.title("Top 15 customers by total revenue")
plt.tight_layout()
plt.show()

In [ ]:
# 11. Metrics collections (pre-aggregated silver-like tables) - sanity check
for coll in ["driver_monthly_metrics", "truck_utilization_metrics"]:
    df = pd.DataFrame(list(db[coll].find().limit(5000)))
    print(f"\n=== {coll} {db[coll].count_documents({}):,} docs ===")
    print(df.head(2).to_string())
    print(df.describe().T.to_string())
    # check month coverage
    print(
        "distinct months:",
        df["month"].nunique(),
        df["month"].min(),
        "->",
        df["month"].max(),
    )
print(
    "\nMaintenance",
    db.maintenance_records.count_documents({}),
    "safety",
    db.safety_incidents.count_documents({}),
)
maint_df = pd.DataFrame(list(db.maintenance_records.find().limit(3000)))
print(maint_df["maintenance_type"].value_counts().to_string())
plt.figure(figsize=(6, 3))
maint_df["total_cost"].hist(bins=40, color="grey", edgecolor="white")
plt.title("maintenance total_cost")
plt.show()

## 12. Findings & Lakehouse Mapping

**Scale:** 14 collections, ~580k docs, 2022-01-01 to 2024-12-31 (fuel to 2025-01). All `updated_at` = 2026-06-16 bulk load.

**Core ER:** `loads 1--1 trips 1--2 delivery_events`, `loads m--1 customers/routes`, `trips m--1 trucks/drivers/trailers`, `safety/maintenance/fuel` hang off trips/trucks. FKs are clean (0 missing).

**Quality flags:**
- Optional FK empty strings: `trips.driver_id` ~2%, `trips.truck/trailer` ~1%, `fuel.driver_id` 2%, `drivers.termination_date` 82% (active drivers) â€” impute to NULL in silver.
- Synthetic bug: `fuel_purchases.location_city='New York'` with `location_state='AZ'` (and likely more city/state mismatches) â€” validate against routes/facilities.
- `accessorial_charges` mostly 0, `fuel_surcharge` skewed, `detention_minutes` long tail (needs capping).
- `on_time_flag` is string `"True"/"False"` not boolean â€” cast in silver.
- No duplicates on business keys (`load_id`, `trip_id` distinct = count).

**Bronze:** raw CDC JSON (`event_id` dedupe, `cluster_time`, `full_document`) per ARCHITECTURE Â§6.2.

**Silver:** explicit schemas (no `inferSchema`), parse `load_date`/`dispatch_date` as dates, cast flags to boolean, emptyâ†’null, dedupe by `(collection, doc_id)` latest `cluster_time`, soft-delete handling.

**Gold (rebuildable, per Â§6.5):**
- `order_facts` â†’ `load_facts` (one row per load+trip denormalized)
- `customer_summary` (LTV, order count, last load, credit terms)
- `order_metrics_daily` â†’ `load_metrics_daily` (revenue, weight, on-time rate, detention)
- `driver_performance` / `truck_utilization` (joins to pre-aggregated metrics + trips)

**Next:** run this notebook before any Spark job; gate silverâ†’gold on reject ratio and row-count sanity (Â§6.4).

In [ ]:
# 13. Export summary JSON for downstream (optional)
summary = {
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "mongo_uri": os.getenv("MONGO_URI"),
    "mongo_db": os.getenv("MONGO_DB"),
    "collections": {r["collection"]: int(r["count"]) for _, r in inv.iterrows()},
    "total_docs": int(inv["count"].sum()),
    "business_dates": {
        "loads": "2022-01-01 to 2024-12-31",
        "fuel_purchases": "to 2025-01-02",
    },
    "quality_flags": [
        "fuel city/state mismatch (New York, AZ)",
        "string boolean flags",
        "empty optional FKs",
        "detention long tail",
    ],
}
out = ROOT / "notebooks" / "eda_summary.json"
out.write_text(json.dumps(summary, indent=2))
print(out)
print(json.dumps(summary, indent=2))